# Ayudantía 14: Repaso Examen 🤓

## Ayudantes 👾
Y sus recomendaciones semanales 🎵

- S1: Josefa Buch
  - [EVERGLOW - LA DI DA](https://open.spotify.com/track/6mIjJONoUMvGPT9Kzrab3L?si=gfzmHRSmQQ-bRHzUVf9cPQ)
- S2: Bastián Pérez
  - [Laura Shigihara - Loonboon](https://youtu.be/lr4vi_XAjQQ?si=o0pRy5p2GfcGO0Cw)
- S3: Daniel Villaseñor
  - _(None)_
- S4: André Angulo
  - [Trueno - RAIN IV](https://www.youtube.com/watch?v=6MLvtbKBNx4&list=RD6MLvtbKBNx4&start_radio=1)
- S5: Clemente Campos
  - [Father John Misty - Holy Shit](https://www.youtube.com/watch?v=7TXdtZLrsZc&list=RD7TXdtZLrsZc&start_radio=1)

Y con aparición especial de:
- Daniela Concha (Profe S5)
  - [グッドバイ -album version-](https://music.youtube.com/watch?v=97omRE2flPU)
- Claudio "Ale" M.D.
  - [Labyrinthus Nocturnus](https://namihaven.bandcamp.com/track/labyrinthus-nocturnus)

## Contenidos 📖
- Serialización y Networking I
- Serialización y Networking II
- Interfaces Gráficas 2
- Tópicos Avanzados (Regex, Grafos, NumPy y Pandas)

# Serialización y Networking I 💻

## Serialización y JSON

Cuando enviamos datos a través de la red, estos deben ser **serializados**. El formato que utilizaremos en este curso es **JSON (JavaScript Object Notation)**.

### ¿Qué es JSON?
JSON es un formato **multilenguaje**, es decir, puede ser leído y generado por prácticamente cualquier lenguaje: Python, Java, C++, JavaScript, Go, etc.
Además de esto es *human-readable*, es decir, puedes leer un archivo .json y entender lo que aparece en este.

### ¿Qué tipos de Python se pueden convertir a JSON?

| Tipo en Python | Representación JSON | ¿Es válido convertirlo? |
|----------------|---------------------|--------------------------|
| `dict`         | objeto JSON         | ✔ Sí |
| `list`, `tuple`| arreglo JSON        | ✔ Sí |
| `str`          | string              | ✔ Sí |
| `int`          | number              | ✔ Sí |
| `float`        | number              | ✔ Sí |
| `bool`         | true/false          | ✔ Sí |
| `None`         | null                | ✔ Sí |

### ¿Qué **no** se puede transformar directamente a JSON?
JSON **no admite** tipos complejos propios de Python, como:
- instancias de clases
- funciones
- sets (`set`)
- bytes
- datetime u otros objetos no estándar

Para enviar esos tipos, primero deben convertirse a valores compatibles: por ejemplo, `datetime → string`, `set → list`, `objeto → dict`.

### Ejemplo de JSON válido
```json
{
  "id": 1,
  "nombre": "Generic Discord MOD",
  "roles": ["admin", "usuario"],
  "activo": true
}
```

### Serialización de objetos personalizados

¿Qué pasa si quiero serializar una instancia de una clase? Como JSON no sabe hacerlo solo, debemos adaptar la información. Una opción rápida es serializar el `__dict__` del objeto, pero eso no nos permite recuperar la instancia original después.

Para personalizar bien la serialización y la deserialización usamos:
- `json.JSONEncoder` (sobrescribiendo el método `default`) para **serializar**.
- el parámetro `object_hook` para **deserializar**.

In [1]:
import json


class Mascota:
    def __init__(self, nombre, edad, especie):
        self.nombre = nombre
        self.edad = edad
        self.especie = especie

    def __repr__(self):
        return f"Mascota({self.nombre}, {self.edad}, {self.especie})"


# Para serializar: heredamos de JSONEncoder y sobrescribimos default
class MascotaEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, Mascota):
            return {"nombre": obj.nombre, "edad": obj.edad, "especie": obj.especie}
        # Para otros tipos, dejamos el comportamiento por defecto
        return super().default(obj)


luna = Mascota("Luna", 8, "Gato")

texto = json.dumps(luna, cls=MascotaEncoder)
print(texto)

{"nombre": "Luna", "edad": 8, "especie": "Gato"}


In [2]:
# Para deserializar: entregamos una función al parámetro object_hook.
# Cada objeto JSON (que llega como dict) pasa por esta función.
def mascota_hook(diccionario):
    return Mascota(diccionario["nombre"], diccionario["edad"], diccionario["especie"])


recuperada = json.loads(texto, object_hook=mascota_hook)
print(recuperada, type(recuperada))

Mascota(Luna, 8, Gato) <class '__main__.Mascota'>


## Networking: conceptos básicos

Para que dos computadores se comuniquen por la red necesitamos identificarlos y ponernos de acuerdo en cómo hablar.

- **Dirección IP**: identifica a un dispositivo dentro de la red (por ejemplo `192.168.0.10`).
- **Puerto**: identifica un proceso o servicio dentro de ese dispositivo (por ejemplo el `5000`). La combinación IP + puerto define el extremo de una comunicación.

Sobre cómo se transmiten los datos, los dos protocolos principales son:

- **TCP** (*Transmission Control Protocol*): orientado a la conexión. Garantiza que los datos lleguen completos y en orden. Es más lento, pero confiable. Es el que usamos en el curso.
- **UDP** (*User Datagram Protocol*): sin conexión. Es más rápido, pero no garantiza la entrega ni el orden. Útil para streaming o juegos.

En la **arquitectura cliente-servidor**, el **servidor** ofrece un recurso o servicio y queda esperando, mientras los **clientes** se conectan a él para pedirlo.

## APIs, Webservices y Endpoints

Una **API** (Application Programming Interface) es un conjunto de reglas que define cómo un software puede comunicarse con otro. Cuando esta comunicación ocurre mediante **HTTP**, hablamos de una **API web** o **webservice**.

En términos simples: **una API web especifica cómo un cliente puede pedir datos a un servidor, cómo debe enviarlos y en qué formato se reciben las respuestas**.

Las APIs suelen organizar su información en **recursos**, cada uno identificado por un **path**. Ejemplos:
- `/usuarios`
- `/apuestas`
- `/casino/aviator`

Para interactuar con esos recursos se utilizan **métodos HTTP** como `GET`, `POST`, `PUT`, `PATCH` o `DELETE`. Estos le indican a la API qué tipo de consulta debe hacer sobre una url definida. En el curso las APIs se crean utilizando el módulo `flask`.

### Endpoints
Están definidos por:

**1. Método HTTP (la acción que quiero realizar)**
- `GET`: obtener datos.
- `POST`: crear un recurso nuevo.
- `PUT`: reemplazar **completamente** un recurso existente (sobrescribe todos los datos).
- `PATCH`: modificar **parcialmente** un recurso existente (solo actualiza los campos enviados).
- `DELETE`: eliminar un recurso.

**2. Path del recurso**
- Asociamos una función (código a ejecutar) cuando se hace una request a cada url.
- Ejemplo: `/usuarios/<id>`
- Puede incluir **parámetros de ruta** (por ejemplo, el ID del usuario).

**3. Parámetros de consulta (query params)**
Son opcionales y van después de `?`: `/usuarios?activo=true&limite=10`. Se pueden obtener con `flask.request.args.get("activo")` y `flask.request.args.get("limite")`.

**4. Cuerpo de la request (body)**
Generalmente se envía en **JSON**, indicando los datos a crear o modificar. Se puede obtener con `flask.request.get_json()`.

**5. Respuesta del servidor (response)**
La respuesta que entrega un servidor suele representarse mediante el objeto **`flask.Response()`**, el cual contiene:
- un **código de estado** (`status=`) HTTP (por ejemplo 200, 201, 400, 404, 500...)
- un **cuerpo** (`response=`), generalmente JSON.
- el **tipo de respuesta** (`content_type=`), generalmente `application/json`.

### Ejemplo de API
API con dos endpoints, con métodos `GET` y `POST` respectivamente.

In [ ]:
from flask import Flask, request, Response, json

app = Flask(__name__)

usuarios = []
id_usuario = [0]


# GET: obtener un usuario
@app.route("/usuarios/<int:user_id>", methods=["GET"])
def obtener_usuario(user_id):
    for usuario in usuarios:
        if usuario["id"] == user_id:
            return Response(
                response=json.dumps(usuario),
                status=200,
                content_type="application/json",
            )


# POST: crear un usuario
@app.route("/usuarios", methods=["POST"])
def crear_usuario():
    body = request.get_json()  # Obtener JSON del body
    id_usuario[0] += 1

    nuevo_usuario = {
        "id": id_usuario[0],
        "nombre": body.get("nombre", "Sin nombre"),
    }
    usuarios.append(nuevo_usuario)

    return Response(
        response=json.dumps({"mensaje": f"Usuario {nuevo_usuario} creado con id: {id_usuario[0]}"}),
        status=201,  # Recurso creado
        content_type="application/json",
    )


if __name__ == "__main__":
    app.run(host="localhost", port=5001)

## Requests HTTP (lado del cliente)

Una **request HTTP** es el mensaje que envía el **cliente** al **servidor** para interactuar con un endpoint. Para este curso las requests se hacen usando el módulo `requests` de Python.

In [ ]:
import requests

En general, una request tiene:

**1. Método HTTP**
La acción que queremos realizar: `GET`, `POST`, `PUT`, `PATCH`, `DELETE`, etc. Se envía como `requests.get(...)`, `requests.post(...)`, etc.

**2. URL completa**
Incluye el **path** del recurso (`"/usuarios/1"`) y, opcionalmente, **parámetros de consulta** (`"/usuarios?activo=true&limite=10"`).

In [ ]:
requests.get("http://localhost:5001/usuarios/1",
             params={"activo": "true", "limite": 10})

**3. Headers (opcional)**
Información adicional sobre la request. Un ejemplo frecuente es `Authorization`, el token de autenticación si la API lo requiere.

In [ ]:
requests.get("http://localhost:5001/usuarios/1",
             headers={"Authorization": "Examen IIC2233"})

**4. Body (opcional)**
Se envía principalmente en `POST`, `PUT` o `PATCH`. En el curso, el body se manda casi siempre en formato **JSON**, con los datos a crear o modificar.

In [ ]:
requests.post("http://localhost:5001/usuarios",
              json={"nombre": "Ada Lovelace"})

Esto automáticamente:
- convierte el dict a JSON,
- agrega el header `Content-Type: application/json`.

### Ejemplo de request POST

In [ ]:
import requests

BASE = "http://localhost:5001"

data = {"nombre": "Skibidi Examen"}
response = requests.post(f"{BASE}/usuarios", json=data)
print(response.status_code)
print(response.json())

### Ejemplo de request GET

In [ ]:
import requests

BASE = "http://localhost:5001"

response = requests.get(f"{BASE}/usuarios/1")
print(response.status_code)
print(response.json())

# Serialización y Networking II 📦

## Pickle

- Serializa objetos a bytes, y no es legible para humanos.
- Puede serializar objetos de Python y clases personalizadas.
- Es específico para Python.
- Puede contener código malicioso.
- Métodos `dumps` y `dump` para serializar objetos. El segundo se usa para serializar directamente a un archivo.
- Métodos `loads` y `load` para deserializar objetos. El segundo se usa para deserializar directamente desde un archivo.

In [3]:
import pickle


class Alumno:

    def __init__(self, nombre, cursos, ayudantias):
        self.nombre = nombre
        self.cursos = cursos
        self.ayudantias = ayudantias

    def __str__(self):
        string = f"Alumno {self.nombre}\nCursos:\n"
        for curso in self.cursos:
            string += f"{curso}\n"
        string += f"Ayudantías:\n"
        for ayudantia in self.ayudantias:
            string += f"{ayudantia}\n"
        return string


alumno = Alumno("clemente",
                ["Diseño y Análisis de Algoritmos",
                 "Fundamentos Matemáticos para Ciencia de Datos",
                 "Algoritmos Paralelos en Computación Científica",
                 "Análisis de Regresión"],
                ["Programación Avanzada",
                 "Introducción a la Programación"])

print(alumno)

Alumno clemente
Cursos:
Diseño y Análisis de Algoritmos
Fundamentos Matemáticos para Ciencia de Datos
Algoritmos Paralelos en Computación Científica
Análisis de Regresión
Ayudantías:
Programación Avanzada
Introducción a la Programación



In [4]:
# lo que nos entrega pickle no es muy legible
alumno_pickle = pickle.dumps(alumno)
print(alumno_pickle)

b'\x80\x04\x95/\x01\x00\x00\x00\x00\x00\x00\x8c\x08__main__\x94\x8c\x06Alumno\x94\x93\x94)\x81\x94}\x94(\x8c\x06nombre\x94\x8c\x08clemente\x94\x8c\x06cursos\x94]\x94(\x8c!Dise\xc3\xb1o y An\xc3\xa1lisis de Algoritmos\x94\x8c.Fundamentos Matem\xc3\xa1ticos para Ciencia de Datos\x94\x8c0Algoritmos Paralelos en Computaci\xc3\xb3n Cient\xc3\xadfica\x94\x8c\x17An\xc3\xa1lisis de Regresi\xc3\xb3n\x94e\x8c\nayudantias\x94]\x94(\x8c\x16Programaci\xc3\xb3n Avanzada\x94\x8c Introducci\xc3\xb3n a la Programaci\xc3\xb3n\x94eub.'


In [5]:
# lo volvemos a cargar
alumno_2 = pickle.loads(alumno_pickle)
print(alumno_2)

Alumno clemente
Cursos:
Diseño y Análisis de Algoritmos
Fundamentos Matemáticos para Ciencia de Datos
Algoritmos Paralelos en Computación Científica
Análisis de Regresión
Ayudantías:
Programación Avanzada
Introducción a la Programación



In [6]:
# el objeto deserializado tiene los mismos datos pero es otro objeto,
# por lo que si lo modificamos el original no se ve afectado
alumno_2.nombre = "evil clemente"
print(alumno.nombre)

clemente


Las funciones `__getstate__` y `__setstate__` permiten personalizar la serialización y deserialización respectivamente. En la segunda puede haber código malicioso, por eso el siguiente código es solo ilustrativo (no lo ejecutamos).

In [ ]:
import pickle


class Alumno:

    def __init__(self, nombre, cursos, ayudantias):
        self.nombre = nombre
        self.cursos = cursos
        self.ayudantias = ayudantias

    def __setstate__(self, state):
        self.__dict__ = state

        print("Me voy a pitiar tu compus!! muahahah")

        for i in range(10**10):
            print(i)


clemente = Alumno("clemente", ["Programación Avanzada"], ["Introducción a la Programación"])

clemente_pickle = pickle.dumps(clemente)
clemente_2 = pickle.loads(clemente_pickle)

## Manejo de bytes

Un byte es un conjunto de 8 bits. Un bit es un 0 ó un 1. En un byte, podemos representar números en binario desde el 0 hasta el 255 (2^8 - 1). Para representar bytes en Python, tenemos dos tipos de dato:

- `bytes`: es una secuencia inmutable, como `str`, que representa una serie de bytes.

In [7]:
# los dos números después de un \x representan un número en hexadecimal, correspondiente a un byte.
mi_byte = b"\x63\x6c\x69\x63"
print(mi_byte)

b'clic'


In [8]:
# esto falla
mi_byte[0] = 1

TypeError: 'bytes' object does not support item assignment

- `bytearray`: es un arreglo de bytes, que es mutable.

In [9]:
# podemos convertir bytes en bytearray
mi_byte_array = bytearray(mi_byte)
print(mi_byte_array)

# podemos modificarlo
mi_byte_array[0] = 67
print(mi_byte_array)

bytearray(b'clic')
bytearray(b'Clic')


Cuando hablamos de un *encoding*, esto es una manera de convertir algo a bytes. Tenemos los siguientes:

- ASCII: se usa para convertir texto en bytes. La versión no extendida contiene sólo 128 caracteres, por lo que no tiene caracteres como la "ñ" o tildes.

In [10]:
# al recorrer el objeto de tipo bytes, nos imprime los números asociados a cada byte
codificado = "juanito".encode("ascii")
for byte in codificado:
    print(byte)

106
117
97
110
105
116
111


In [11]:
# decodificamos
print(codificado.decode("ascii"))

juanito


- UTF-8: el estándar moderno. Contiene muchísimos caracteres, y los primeros 128 son los mismos que en ASCII, por lo que cualquier string ASCII puede ser codificado y decodificado en formato UTF-8.

In [12]:
# por fin podemos codificar la ñ!!
print("ñ".encode("utf-8"))

# también símbolos muy raros
print("电脑".encode("utf-8"))
print("компьютер".encode("utf-8"))

b'\xc3\xb1'
b'\xe7\x94\xb5\xe8\x84\x91'
b'\xd0\xba\xd0\xbe\xd0\xbc\xd0\xbf\xd1\x8c\xd1\x8e\xd1\x82\xd0\xb5\xd1\x80'


In [13]:
# si fue codificado como ascii, podemos decodificarlo como utf-8
ascii = "holacabros".encode("ascii")
print(ascii.decode("utf-8"))

holacabros


In [14]:
# pero no siempre podemos hacerlo al revés
utf = "ऋ".encode("utf-8")
print(utf.decode("ascii"))

UnicodeDecodeError: 'ascii' codec can't decode byte 0xe0 in position 0: ordinal not in range(128)

## Sockets

Un *socket* es un elemento que nos permite transmitir datos desde un dispositivo a otro. Un servidor y un cliente lo usan de la siguiente forma:

- Servidor
  - Crea el socket asociado al servidor.
  - Hace `bind` con su dirección IP y el puerto desde donde escuchará.
  - Hace `listen` para empezar a escuchar. Se le puede indicar una cantidad máxima de clientes conectados.
  - Hace `accept`, que queda esperando hasta que se conecte un nuevo cliente.
  - Hace `recv` y `send` con **el socket del cliente** para recibir y mandar información como bytes.
- Cliente
  - Crea el socket asociado al cliente.
  - Hace `connect` con la dirección IP y el puerto **del servidor**.
  - Hace `recv` y `send` con **su socket** para recibir y mandar información como bytes.

Para un servidor es fundamental el uso de *threads* en conjunto con networking para mantener la comunicación simultánea con todos los clientes. Asimismo, el cliente generalmente tiene un *thread* que se encarga de escuchar al servidor en todo momento.

- Arquitectura P2P: es un tipo de arquitectura alternativa a Cliente-Servidor. Cada usuario es a la vez cliente y servidor y se comunican de a pares (*peer to peer*). Es una arquitectura descentralizada ideal para cosas como transferencia de archivos.

El código de servidor y cliente que sigue es ilustrativo: cada uno se corre como un archivo aparte, ya que `accept` y `recv` bloquean la ejecución.

In [ ]:
# servidor_tcp.py
import socket

# AF_INET: IPv4   |   SOCK_STREAM: TCP
servidor = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
servidor.bind(("localhost", 5000))
servidor.listen()

print("Esperando conexiones...")
socket_cliente, direccion = servidor.accept()   # bloquea hasta que llega un cliente

mensaje = socket_cliente.recv(1024)              # recibe bytes
print("Recibí:", mensaje.decode("utf-8"))

socket_cliente.send("Hola cliente".encode("utf-8"))
socket_cliente.close()
servidor.close()

In [ ]:
# cliente_tcp.py
import socket

cliente = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
cliente.connect(("localhost", 5000))             # se conecta al servidor

cliente.send("Hola servidor".encode("utf-8"))    # envía bytes
respuesta = cliente.recv(1024)
print("El servidor respondió:", respuesta.decode("utf-8"))

cliente.close()

# Interfaces Gráficas 2 🖼️

## QThreads

Una `QThread` es como una `Thread` normal del módulo `threading`, pero es adicionalmente un objeto de PyQt, por lo que puede contener señales. Ofrece su propia versión de los *locks*, llamados `QMutex`, que se bloquean y desbloquean con los métodos `lock` y `unlock` respectivamente.

El código siguiente es ilustrativo (necesita una aplicación de PyQt corriendo).

In [ ]:
from PyQt6.QtCore import QThread, pyqtSignal
import time


class Trabajadora(QThread):
    # señal que le avisa el progreso a la interfaz
    progreso = pyqtSignal(int)

    def run(self):
        for i in range(5):
            time.sleep(1)
            self.progreso.emit(i)    # emitimos la señal con un argumento


# En la ventana conectaríamos la señal a un método que actualiza un QLabel:
# self.trabajadora = Trabajadora()
# self.trabajadora.progreso.connect(self.actualizar_label)
# self.trabajadora.start()

## Integración con Networking

Una integración de interfaces gráficas con networking se vería de la siguiente manera:

- Servidor: generalmente el servidor no tiene una interfaz gráfica. Se comporta de la misma manera que vimos anteriormente.
- Cliente:
  - Frontend: se encarga de los elementos visuales.
  - Backend: se encarga de la lógica y además **de la comunicación con el servidor**.

Si es que, por ejemplo, tenemos una partida multijugador de gato, y un cliente quiere hacer una jugada, la comunicación es de la siguiente forma:

- Un cliente apreta la casilla donde quiere hacer la jugada. El frontend le comunica esto al backend con una **señal**.
- El backend revisa si la jugada es válida, y si lo es, se la envía al servidor mediante su **socket**.
- El servidor procesa la jugada correspondientemente, y se la informa a **ambos jugadores** mediante sus **sockets**.
- El backend de ambos jugadores recibe la jugada, y se la informa al frontend mediante una **señal**, para que finalmente actualice la interfaz.

## PyQt Misceláneo

- El método `isAutoRepeat` lo podemos usar dentro de un `keyPressEvent` para revisar si una tecla fue pulsada solo una vez (`False`) o fue mantenida (`True`).
- La clase `QMediaPlayer` nos permite reproducir archivos `.mp3`.
- La clase `QSoundEffect` nos permite reproducir archivos `.wav`.

# Tópicos Avanzados 🧠

## Regex

Las expresiones regulares nos permiten buscar patrones dentro de una cadena de texto. Estos patrones pueden ser tan restrictivos como nosotros queramos, por lo que es especialmente útil cuando queremos una "secuencia similar a algo".

### Metacaracteres
- `[]`: permite agrupar clases de caracteres. Añadiendo un `-` podemos especificar rangos.
- `.`: hace match con cualquier caracter.
- `\`: indica que un metacaracter presente es texto literal.
- `\d`: cualquier dígito, su opuesto es `\D`.
- `\w`: cualquier letra o dígito, su opuesto es `\W`.
- `\s`: cualquier espacio en blanco (incluye espacios y tabulaciones), su opuesto es `\S`.

### Cuantificadores
- `+`: el patrón anterior se repite 1 o más veces.
- `*`: el patrón anterior se repite 0 o más veces.
- `?`: puede o no estar. Si está, es exactamente una vez.
- `{n, m}`: el patrón anterior se repite entre n y m veces.

### Aserciones
- `^`: identifica el inicio del string.
- `$`: identifica el final del string.
- `A | B`: permite reconocer el patrón A o el patrón B.
- `( )`: permite reconocer grupos de patrones.
- `(?P<Nombre> )`: permite asignar el `Nombre` al patrón reconocido, para acceder a él por medio de `Nombre`.

### Módulo `re`
- `match()`: verifica si un substring cumple con la expresión regular a partir del inicio del string.
- `fullmatch()`: verifica si el string completo cumple con la expresión regular.
- `search()`: verifica si algún substring cumple con la expresión regular.
- `sub()`: permite reemplazar un patrón por otra secuencia de caracteres en un string.
- `split()`: permite separar un string de acuerdo a un patrón.

In [15]:
import re

# Patrón sencillo de correo: algo@algo.algo
patron = r"[\w.]+@[\w]+\.[a-z]+"

texto = "escribe a juan@uc.cl o a maria.paz@gmail.com"
print(re.search(patron, texto))     # primer match (un objeto Match)
print(re.findall(patron, texto))    # todos los matches

<re.Match object; span=(10, 20), match='juan@uc.cl'>
['juan@uc.cl', 'maria.paz@gmail.com']


## Grafos

Conjunto de **nodos** (o vértices) y **aristas**.

Grafo **dirigido**: si se llega del nodo A al nodo B, no necesariamente se puede llegar del nodo B al nodo A.

Grafo **no dirigido**: si se llega del nodo A al nodo B, entonces se llega del nodo B al nodo A.

### Representación en Listas de Adyacencia

In [16]:
grafo_no_dirigido = {
    1: [2],
    2: [1, 3],
    3: [2, 4, 5],
    4: [3, 5],
    5: [3, 4]
}

grafo_dirigido = {
    1: [2],
    2: [3],
    3: [2, 4, 5],
    4: [5],
    5: []
}

### Representación en Matriz de Adyacencia

In [17]:
grafo_no_dirigido = [[0, 1, 0, 0, 0],
                     [1, 0, 1, 0, 0],
                     [0, 1, 0, 1, 1],
                     [0, 0, 1, 0, 1],
                     [0, 0, 1, 1, 0]]

grafo_dirigido    = [[0, 1, 0, 0, 0],
                     [0, 0, 1, 0, 0],
                     [0, 1, 0, 1, 1],
                     [0, 0, 0, 0, 1],
                     [0, 0, 0, 0, 0]]

### Aristas con pesos

Agrega el costo que toma pasar por una arista. Podemos modelarlo así:

In [18]:
# Listas de Adyacencia con pesos
grafo_no_dirigido = {
    1: [2],
    2: [1, 3],
    3: [2, 4, 5],
    4: [3, 5],
    5: [3, 4]
}

pesos_no_dirigido = {
    (1, 2): 2,
    (2, 3): 4,
    (3, 4): 3,
    (3, 5): 1,
    (4, 5): 5,
    # Se añaden las aristas inversas para facilitar la consulta
    (2, 1): 2,
    (3, 2): 4,
    (4, 3): 3,
    (5, 3): 1,
    (5, 4): 5
}

In [19]:
# Matriz de Adyacencia con pesos (0 = no hay arista)
grafo_no_dirigido = [[0, 2, 0, 0, 0],
                     [2, 0, 1, 0, 0],
                     [0, 1, 0, 5, 7],
                     [0, 0, 5, 0, 4],
                     [0, 0, 7, 4, 0]]

## Recorrido de grafos

### BFS (Breadth First Search)
La búsqueda en amplitud recorre el grafo por niveles, utilizando una cola que agrega los nodos (FIFO). Ideal para búsqueda de caminos más cortos.

### DFS (Depth First Search)
La búsqueda en profundidad recorre el grafo por un camino hasta que no queden más nodos, utilizando un stack que agrega los nodos (LIFO).

In [20]:
from collections import deque


def dfs(grafo, inicio):
    # Vamos a mantener un set con los nodos visitados.
    visitados = set()
    # El stack de siempre, comienza desde el nodo inicio.
    stack = [inicio]
    orden = []
    while len(stack) > 0:
        vertice = stack.pop()
        # Detalle clave: si ya visitamos el nodo, no hacemos nada.
        if vertice in visitados:
            continue
        visitados.add(vertice)
        orden.append(vertice)
        # Agregamos los vecinos al stack si es que no han sido visitados.
        for vecino in grafo[vertice]:
            if vecino not in visitados:
                stack.append(vecino)
    return orden


def bfs(grafo, inicio):
    visitados = set()
    # Usamos una cola, comienza desde el nodo inicio.
    cola = deque([inicio])
    orden = []
    while len(cola) > 0:
        vertice = cola.popleft()
        if vertice in visitados:
            continue
        visitados.add(vertice)
        orden.append(vertice)
        for vecino in grafo[vertice]:
            if vecino not in visitados:
                cola.append(vecino)
    return orden


grafo_demo = {1: [2], 2: [1, 3], 3: [2, 4, 5], 4: [3, 5], 5: [3, 4]}
print("DFS:", dfs(grafo_demo, 1))
print("BFS:", bfs(grafo_demo, 1))

DFS: [1, 2, 3, 5, 4]
BFS: [1, 2, 3, 4, 5]


## NumPy

Un arreglo (*array*) es un espacio de memoria físico. Se asemeja a las listas de Python, pero es menos flexible: un arreglo no puede tener tipos de datos distintos. Están programados en C, por lo que son más veloces que las listas y están optimizados para operaciones matriciales.

Algunas funciones de NumPy:
- `arange()`: similar a `range()`, pero devuelve un array.
- `linspace()`: genera números equiespaciados en un intervalo.
- `zeros()` y `ones()`: crean arrays de ceros o unos.
- `eye()`: crea una matriz identidad.
- `random`: genera arrays con números aleatorios.

### Operaciones con escalares

In [21]:
import numpy as np

a = np.array([1, 2, 3])

print("Suma con escalar:", a + 10)
print("Multiplicación con escalar:", a * 3)

Suma con escalar:

 [11 12 13]
Multiplicación con escalar: [3 6 9]


### Operaciones aritméticas

In [22]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("Suma:", a + b)
print("Resta:", a - b)
print("Multiplicación:", a * b)
print("División:", a / b)
print("Potencia:", a ** 2)

array = np.array([0, np.pi / 2, np.pi])

print("Seno:", np.sin(array))
print("Coseno:", np.cos(array))
print("Exponencial:", np.exp(array))
print("Logaritmo:", np.log(array + 1))  # para evitar log(0)

Suma: [5 7 9]
Resta: [-3 -3 -3]
Multiplicación: [ 4 10 18]
División: [0.25 0.4  0.5 ]
Potencia: [1 4 9]
Seno: [0.0000000e+00 1.0000000e+00 1.2246468e-16]
Coseno: [ 1.000000e+00  6.123234e-17 -1.000000e+00]
Exponencial: [ 1.          4.81047738 23.14069263]
Logaritmo: [0.         0.94421571 1.42108041]


### Indexaciones y Slicing

In [23]:
array = np.array([10, 20, 30, 40, 50])

print("Elemento en posición 0:", array[0])
print("Último elemento:", array[-1])
print("Slice de 1 a 3:", array[1:4])

array[0] = 100
print("Array modificado:", array)

Elemento en posición 0: 10
Último elemento: 50
Slice de 1 a 3: [20 30 40]
Array modificado: [100  20  30  40  50]


### Concatenación y División

In [24]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

array = np.array([1, 2, 3, 4, 5, 6])
split_arrays = np.split(array, 3)
print("Arrays divididos:", split_arrays)

concatenado = np.concatenate((a, b))
print("Arrays concatenados:", concatenado)

Arrays divididos: [array([1, 2]), array([3, 4]), array([5, 6])]
Arrays concatenados: [1 2 3 4 5 6]


### Análisis y Estadística

In [25]:
array = np.array([1, 2, 3, 4, 5])

print("Suma:", np.sum(array))
print("Media:", np.mean(array))
print("Mediana:", np.median(array))
print("Desviación estándar:", np.std(array))
print("Varianza:", np.var(array))
print("Valor máximo:", np.max(array))
print("Valor mínimo:", np.min(array))
print("Índice del valor máximo:", np.argmax(array))

Suma: 15
Media: 3.0
Mediana: 3.0
Desviación estándar: 1.4142135623730951
Varianza: 2.0
Valor máximo: 5
Valor mínimo: 1
Índice del valor máximo: 4


## Pandas

Pandas se manipula con DataFrames, una estructura de datos similar a tablas de Excel. En Python se manipulan parecido a diccionarios, donde cada llave es una columna y el valor es una lista con los datos.

In [26]:
import pandas as pd

datos_peliculas = {
    "Título": ["Frozen", "Toy Story", "The Lion King", "Moana", "Coco"],
    "Año": [2013, 1995, 1994, 2016, 2017],
    "Género": ["Animación", "Animación", "Animación", "Animación", "Animación"],
    "Duración": [102, 81, 88, 107, 105],
    "Calificación": [7.4, 8.3, 8.5, 7.6, 8.4],
}

df_peliculas = pd.DataFrame(datos_peliculas)
df_peliculas

,Título,Año,Género,Duración,Calificación
0,Frozen,2013,Animación,102,7.4
1,Toy Story,1995,Animación,81,8.3
2,The Lion King,1994,Animación,88,8.5
3,Moana,2016,Animación,107,7.6
4,Coco,2017,Animación,105,8.4


### Métodos

In [27]:
# Primeras 2 filas
df_peliculas.head(2)

,Título,Año,Género,Duración,Calificación
0,Frozen,2013,Animación,102,7.4
1,Toy Story,1995,Animación,81,8.3


In [28]:
# Estadísticas de las columnas numéricas
df_peliculas.describe()

,Año,Duración,Calificación
count,5.000000,5.000000,5.000000
mean,2007.000000,96.600000,8.040000
std,11.510864,11.458621,0.502991
min,1994.000000,81.000000,7.400000
25%,1995.000000,88.000000,7.600000
50%,2013.000000,102.000000,8.300000
75%,2016.000000,105.000000,8.400000
max,2017.000000,107.000000,8.500000


In [29]:
# Acceder a los valores de una columna
df_peliculas["Año"]

0    2013
1    1995
2    1994
3    2016
4    2017
Name: Año, dtype: int64

In [30]:
# Rango por índices (iloc) y por labels (loc)
print(df_peliculas.iloc[1:3])
print()
print(df_peliculas.loc[:, "Género":"Calificación"])

          Título   Año     Género  Duración  Calificación
1      Toy Story  1995  Animación        81           8.3
2  The Lion King  1994  Animación        88           8.5

      Género  Duración  Calificación
0  Animación       102           7.4
1  Animación        81           8.3
2  Animación        88           8.5
3  Animación       107           7.6
4  Animación       105           8.4


In [31]:
# Filtrado por valores
df_peliculas[df_peliculas["Calificación"] > 8]

,Título,Año,Género,Duración,Calificación
1,Toy Story,1995,Animación,81,8.3
2,The Lion King,1994,Animación,88,8.5
4,Coco,2017,Animación,105,8.4


In [32]:
# Nuevos registros
nueva_pelicula = {
    "Título": "Star Wars: Episodio III - La Venganza de los Sith",
    "Año": 2005,
    "Género": "Ciencia Ficción/Acción",
    "Duración": 140,
    "Calificación": 7.6,
}

df_nueva_pelicula = pd.DataFrame([nueva_pelicula])
df_peliculas = pd.concat([df_peliculas, df_nueva_pelicula], ignore_index=True)
df_peliculas

,Título,Año,Género,Duración,Calificación
0,Frozen,2013,Animación,102,7.4
1,Toy Story,1995,Animación,81,8.3
2,The Lion King,1994,Animación,88,8.5
3,Moana,2016,Animación,107,7.6
4,Coco,2017,Animación,105,8.4
5,Star Wars: Episodio III - La Venganza de los Sith,2005,Ciencia Ficción/Acción,140,7.6


# Preguntas de Desarrollo

## DCCarioca.1

DCCarioca es un torneo online de Carioca (multijugador) organizado por el DCC. Basándose en una arquitectura cliente/servidor, indique cuáles son las clases que necesitaría para programar el juego, indicando para cada una si pertenece al cliente o servidor y a su respectivo frontend o backend.

**Respuesta:**

---

## DCCarioca.2

En DCCarioca, cuando un jugador bota una carta, el resto de los jugadores tienen la oportunidad de adivinarla. Si un jugador adivina la pinta o el valor de la carta (sólo uno de los dos), se le dan 10 puntos extra. Si adivina tanto la pinta como el valor, se le entrega una carta adicional a su mazo. Considere que el puntaje de un jugador es visible para todo otro jugador en tiempo real, pero el mazo sólo es visible para el mismo jugador.

Considere que tiene tres jugadores (A, B y C) y muestre este proceso indicando los mensajes que se deberían mandar entre los sockets, tomando en cuenta el formato de éstos, qué clase los manda y a quién. No es necesario mostrar la parte de interfaces gráficas.

**Respuesta:**

---

## DCCarioca.3

Se debe implementar un proceso de serialización que transforme el string (en formato JSON) del mensaje a bytes y luego lo encripte usando el siguiente protocolo:

1. El string original se transforma a bytes.
2. Los bytes del mensaje se dividen en chunks de 124 bytes.
3. Si el último chunk tiene menos de 124 bytes, se rellena con bytes cero hasta completar 124 bytes.
4. A cada chunk se le agrega al inicio el número del chunk codificado en 4 bytes.
5. Antes de enviar los chunks, se envía el largo original del mensaje, también codificado en 4 bytes.
6. Los chunks se envían en orden aleatorio.
7. Al recibir el mensaje, se debe reconstruir el orden original usando el número de cada chunk.
8. Finalmente, se debe eliminar el relleno agregado y recuperar el string original.

A partir de lo anterior, escriba el pseudocódigo de la función `serializar`, que reciba un string y retorne los bytes encriptados, y la función `deserializar`, que haga el proceso opuesto.

**Respuesta:**

---


## Verdadero y Falso
Indique si las siguientes aseveraciones son verdaderas o falsas, justificando en cada caso.

1. En la implementación de DFS y BFS, lo único que cambia es que la primera usa una cola para almacenar los nodos por visitar y la segunda un stack.

**Respuesta:**

---

2. Si tenemos un DataFrame donde una de las columnas es `"edad"`, la expresión `df[df["edad"] > 18]` es correcta y modifica el DataFrame original para dejar solo las filas donde la columna `"edad"` es mayor a 18.

**Respuesta:**

---

3. Los arrays de numpy permiten hacer cálculos vectoriales mucho más rápidos que las listas de python, pero las listas son más eficientes en uso de memoria.

**Respuesta:**

---

4. En una arquitectura cliente/servidor usando TCP, el cliente normalmente debe ejecutar `listen()` antes de poder aceptar conexiones entrantes.

**Respuesta:**

---

5. Para poder establecer una conexión, el cliente debe conocer la dirección IP y el puerto del servidor, mientras que el servidor debe conocer la dirección IP y puerto del cliente.

**Respuesta:**

---

6. En la comunicación mediante sockets se pueden enviar `strings`, siempre que estén en formato JSON.

**Respuesta:**

---

# Solución

## DCCarioca.1

- Ventana (cliente, frontend): clase encargada de mostrar los elementos gráficos del cliente.
- Backend (cliente, backend): clase encargada de manejar el backend del cliente, hacer la conexión con el servidor y manejar la lógica del juego (opcional).
- Servidor (servidor, backend): clase encargada de hacer la conexión con todos los clientes y manejar la lógica del juego.

---

## DCCarioca.2

Primero, el cliente A (`Backend` de A) le informa al servidor (`Servidor`) de su carta, con un mensaje del estilo:

```python
msg = {
    "accion": "botar carta",
    "pinta": pinta,
    "valor": valor
} 
```
Notemos que no es necesario incluir el nombre del jugador en el mensaje, ya que asumimos que el servidor tiene una thread para comunicarse con cada cliente. Luego de esto, el `Servidor` le debe informar al jugador B y C (`Backend` de B y C) que deben adivinar una carta, mediante el siguiente mensaje:

```python
msg = {
    "accion": "adivinar carta"
}
```

Ahora cada uno de los jugadores (`Backend` B y C) deben informar de su adivinanza al `Servidor`, mandando el siguiente mensaje:

```python
msg = {
    "accion": "adivinar carta",
    "pinta": pinta,
    "valor": valor
}
```

Ahora el servidor revisa las adivinanzas de cada jugador. Si uno de los jugadores (asumamos el jugador B) adivina solo la pinta o el valor, el `Servidor` le manda el siguiente mensaje a **todos** los jugadores (`Backend` A, B y C):

```python
msg = {
    "accion": "actualizar puntaje",
    "jugador": "B",
    "aumento": 10
}
```

En cambio, si el jugador le acertó tanto a la pinta como al valor, el `Servidor` le manda el siguiente mensaje únicamente al `Backend` de B:

```python
msg = {
    "accion": "nueva carta",
    "pinta": pinta,
    "valor": valor
}
```

Nótese también que todos los mensajes deben ser convertidos a bytes de la siguiente forma:

```python
msg_bytes = json.dumps(msg).encode(encoding='UTF-8')
```

---

## DCCarioca.3

```python
def serializar(mensaje: str) -> bytes:

    mensaje_bytes = mensaje.encode("utf-8")

    largo_original = len(mensaje_bytes)

    bytes_largo = largo_original.to_bytes(4, byteorder="big")

    chunks = []

    cant_chunks = ceil(largo_original / 124)
    numero_chunk = 0

    for i in range(0, largo_original):

        chunk = mensaje_bytes[i * 124 : (i + 1) * 124]

        if len(chunk) < 124:
            cantidad_relleno = 124 - len(chunk)
            chunk = chunk + b"\x00" * cantidad_relleno

        bytes_numero_chunk = i.to_bytes(4, byteorder="big")

        chunk_codificado = bytes_numero_chunk + chunk

        chunks.append(chunk_codificado)


    mezclar_aleatoriamente(chunks)

    mensaje_serializado = bytes_largo

    for chunk in chunks:
        mensaje_serializado += chunk

    return mensaje_serializado
```

```python
def deserializar(datos: bytes) -> str:

    bytes_largo = datos[0:4]
    largo_original = int.from_bytes(bytes_largo, byteorder="big")

    datos_chunks = datos[4:]

    chunks = {}
    cantidad_chunks = datos_chunks / 128

    for i in range(0, cantidad_chunks):

        chunk_codificado = datos_chunks[i * 128 : (i + 1) * 128]

        bytes_numero_chunk = chunk_codificado[0:4]
        contenido_chunk = chunk_codificado[4:128]

        numero_chunk = int.from_bytes(bytes_numero_chunk, byteorder="big")

        chunks[numero_chunk] = contenido_chunk


    mensaje_bytes = b""

    chunks = ordenar_llaves_de_menor_a_mayor(chunks)

    for numero_chunk in chunks:

        mensaje_bytes += chunks[numero_chunk]


    mensaje_bytes = mensaje_bytes[0:largo_original]

    mensaje = mensaje_bytes.decode("utf-8")

    return mensaje
```

---

## Verdadero y Falso

1. Falso. Es al revés, DFS usa un stack para recorrer los nodos y BFS una cola.

---

2. Falso. La expresión es correcta y filtra de esa manera, pero no modifica el dataframe original sino que crea uno nuevo.

---

3. Falso. La primera parte es verdad, pero la segunda no, los arrays de numpy son más eficientes en memoria ya que guardan los elementos en espacios contiguos.

---
 
4. Falso. El servidor es quien debe hacer `listen()`.

---

5. Falso. El cliente necesita saber la dirección IP y el puerto del servidor, pero el servidor no necesita saber los datos del cliente para establecer la conexión.

---

6. Falso. Mediante sockets sólo se pueden enviar `bytes`. Si tenemos un string en formato JSON, primero debemos pasarlo por un encoding para mandarlo por un socket.